# **Project Name** — Voyage Analytics — Hotel Recommendation System

##### **Project Type** — EDA / Unsupervised Learning / Recommendation
##### **Contribution** — Individual
##### **Team Member 1 — Harsh Paul**

# **Project Summary**

This notebook produces hotel suggestions from completed stays. It creates
an implicit-feedback user–hotel matrix, uses Truncated SVD to learn latent
preferences, and evaluates a top-five ranking against each user’s held-out latest
hotel.

This standalone notebook uses the supplied travel_capstone data, follows the
provided EDA submission template, and is ready to run in Google Colab. It explains
the data checks, modelling choices, evaluation approach, and business meaning of
the findings rather than only printing code output.

# **GitHub Link**

[https://github.com/harshpaul1999-tech/Voyage-Analytics](https://github.com/harshpaul1999-tech/Voyage-Analytics)

# **Problem Statement**

#### **Define Your Business Objective?**

Recommend unseen hotels from previous booking behaviour. The model must
hide the latest interaction before training so HitRate@5 represents a future-like
ranking task rather than a lookup of a previously seen booking.

All model inputs are limited to information available at prediction time. The
holdout logic is kept separate from fitting so reported results reflect unseen
examples rather than memorisation.

# **General Guidelines**

- Run every cell from top to bottom in a fresh Colab runtime.
- The setup cell clones the public GitHub repository automatically in Colab.
- The charts and metrics are calculated from the supplied data, not typed in.
- A fixed random seed is used when a split or model needs randomness.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
from pathlib import Path
import sys
if "google.colab" in sys.modules:
    if not Path("/content/Voyage-Analytics").exists():
        !git clone --depth 1 https://github.com/harshpaul1999-tech/Voyage-Analytics.git /content/Voyage-Analytics
    %cd /content/Voyage-Analytics

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

sns.set_theme(style="whitegrid", palette="deep")
RANDOM_STATE = 42
DATA_DIR = Path("data/raw")

### Dataset Loading

In [ ]:
hotels = pd.read_csv(DATA_DIR / "hotels.csv")
hotels["date"] = pd.to_datetime(hotels["date"], format="%m/%d/%Y", errors="coerce")
print("Loaded hotels:", hotels.shape)

### Dataset First View

In [ ]:
display(hotels.head())

### Dataset Rows & Columns count

In [ ]:
print(hotels.shape)

### Dataset Information

In [ ]:
hotels.info()

#### Duplicate Values

In [ ]:
print('Duplicate rows:', hotels.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
display(hotels.isna().sum().sort_values(ascending=False).to_frame('missing_values'))

### What did you know about your dataset?

A hotel stay is treated as implicit positive feedback: it records a
selected hotel but not an explicit rating. Hotel name and place together form the
item identity, while date orders interactions for evaluation.

## ***2. Understanding Your Variables***

### Variables Description

| Variable | Meaning | Model role |
|---|---|---|
| userCode | Traveller identifier | Matrix row |
| name, place | Hotel and location | Combined item key |
| days | Stay length | Small positive-affinity adjustment |
| price, total | Booking economics | Exploratory context |
| date | Booking date | Last-interaction holdout |
| travelCode | Travel identifier | Traceability only |

### Check Unique Values for each variable.

In [ ]:
display(hotels.nunique().sort_values(ascending=False).to_frame('unique_values'))

## ***3. Data Wrangling***

### Data Wrangling Code

In [ ]:
rec_df = hotels.dropna(subset=['userCode', 'name', 'place', 'date', 'days']).copy()
rec_df['hotel_key'] = rec_df['name'].astype(str).str.strip() + ' | ' + rec_df['place'].astype(str).str.strip()
rec_df = rec_df.sort_values(['userCode', 'date']).reset_index(drop=True)
rec_df['affinity'] = 1.0 + .10 * np.clip(rec_df['days'].astype(float) - 1, 0, 14)
print("Rows retained:", len(rec_df), "| hotel items:", rec_df['hotel_key'].nunique())

### What all manipulations have you done and insights you found?

Hotel name and place are combined because names may repeat across locations. Every
completed stay is a positive interaction with base affinity 1.0. Longer stays add
a light, capped bonus, so a single unusually long stay does not overpower repeated
booking evidence.

## ***4. Data Visualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1: Hotel booking counts

##### 1. Why did you pick the specific chart?
A bar chart shows observed item popularity.

##### 2. What is/are the insight(s) found from the chart?
It identifies hotels with more historical interactions.

##### 3. Will the gained insights help creating a positive business impact?
Popularity provides a transparent cold-start fallback.

In [ ]:
rec_df['hotel_key'].value_counts().sort_values().plot(kind='barh', title='Hotel booking counts'); plt.show()

#### Chart - 2: Bookings by destination

##### 1. Why did you pick the specific chart?
Location is a core hotel decision context.

##### 2. What is/are the insight(s) found from the chart?
It shows destinations with the most stays.

##### 3. Will the gained insights help creating a positive business impact?
It helps travel teams understand demand concentration.

In [ ]:
sns.countplot(data=rec_df, y='place', order=rec_df['place'].value_counts().index); plt.title('Bookings by destination'); plt.show()

#### Chart - 3: Stay-length distribution

##### 1. Why did you pick the specific chart?
Days determines the light affinity adjustment.

##### 2. What is/are the insight(s) found from the chart?
It shows whether historical stays are mostly short or long.

##### 3. Will the gained insights help creating a positive business impact?
It validates using a cap on the length contribution.

In [ ]:
sns.histplot(data=rec_df, x='days', discrete=True); plt.title('Stay-length distribution'); plt.show()

#### Chart - 4: Daily price distribution

##### 1. Why did you pick the specific chart?
Price provides budget context for hotel suggestions.

##### 2. What is/are the insight(s) found from the chart?
It shows typical and high daily rates.

##### 3. Will the gained insights help creating a positive business impact?
It motivates a future budget-aware recommender filter.

In [ ]:
sns.histplot(data=rec_df, x='price', bins=30, kde=True); plt.title('Daily hotel-price distribution'); plt.show()

#### Chart - 5: Total booking-value distribution

##### 1. Why did you pick the specific chart?
Total combines price and stay length.

##### 2. What is/are the insight(s) found from the chart?
It shows the scale of historical spend.

##### 3. Will the gained insights help creating a positive business impact?
It supports later traveller-value analysis.

In [ ]:
sns.histplot(data=rec_df, x='total', bins=30, kde=True); plt.title('Total booking value'); plt.show()

#### Chart - 6: Daily price versus total

##### 1. Why did you pick the specific chart?
A scatterplot checks economic consistency.

##### 2. What is/are the insight(s) found from the chart?
It shows that total varies with both price and length.

##### 3. Will the gained insights help creating a positive business impact?
It supports future value-aware filtering.

In [ ]:
sample = rec_df.sample(min(5000, len(rec_df)), random_state=RANDOM_STATE); sns.scatterplot(data=sample, x='price', y='total', alpha=.25); plt.title('Daily price versus total'); plt.show()

#### Chart - 7: Stay days versus total

##### 1. Why did you pick the specific chart?
This isolates the other cost component.

##### 2. What is/are the insight(s) found from the chart?
It shows the expected relationship between length and booking value.

##### 3. Will the gained insights help creating a positive business impact?
It supports keeping days as a weak preference weight, not a ranking target.

In [ ]:
sns.scatterplot(data=sample, x='days', y='total', alpha=.25); plt.title('Stay days versus total'); plt.show()

#### Chart - 8: Average price by place

##### 1. Why did you pick the specific chart?
A bar chart compares destination price context.

##### 2. What is/are the insight(s) found from the chart?
It identifies places with higher observed daily rates.

##### 3. Will the gained insights help creating a positive business impact?
It can support budget-aware ranking constraints.

In [ ]:
rec_df.groupby('place')['price'].mean().sort_values().plot(kind='barh', title='Average price by place'); plt.show()

#### Chart - 9: Unique travellers by hotel

##### 1. Why did you pick the specific chart?
Unique-user count distinguishes broad use from repeat use.

##### 2. What is/are the insight(s) found from the chart?
It shows which hotel items attract the most different travellers.

##### 3. Will the gained insights help creating a positive business impact?
It gives a second popularity signal for cold start.

In [ ]:
rec_df.groupby('hotel_key')['userCode'].nunique().sort_values().plot(kind='barh', title='Unique travellers by hotel'); plt.show()

#### Chart - 10: Monthly booking activity

##### 1. Why did you pick the specific chart?
A time chart tests for seasonality.

##### 2. What is/are the insight(s) found from the chart?
It shows observed booking counts over months.

##### 3. Will the gained insights help creating a positive business impact?
It suggests that future recommendations can incorporate season and availability.

In [ ]:
monthly = rec_df.groupby(rec_df['date'].dt.month).size().reset_index(name='bookings'); sns.lineplot(data=monthly, x='date', y='bookings', marker='o'); plt.xlabel('month'); plt.title('Monthly hotel bookings'); plt.show()

## **5. Solution to Business Objective**

The recommender returns unseen hotels ranked by learned affinity. When history is
not available, the production API fills remaining slots with popular hotels so a
new user receives a useful, transparent fallback.

## ***6. Feature Engineering & Data Pre-processing***

The training input is a user-by-hotel affinity matrix. Rows are users, columns are
combined hotel/place items, and zero means no observed stay. The final stay for
eligible users is removed before matrix construction to prevent leakage.

## ***7. Model Implementation***

### Implicit-feedback Truncated SVD recommender

In [ ]:
eligible = rec_df.groupby('userCode').filter(lambda rows: len(rows) >= 2)
holdout_index = eligible.groupby('userCode').tail(1).index
test_events = rec_df.loc[holdout_index].copy()
train_events = rec_df.drop(index=holdout_index).copy()

interaction_matrix = train_events.pivot_table(
    index='userCode', columns='hotel_key', values='affinity', aggfunc='sum', fill_value=0.0
).sort_index().sort_index(axis=1)
n_components = max(1, min(12, interaction_matrix.shape[0]-1, interaction_matrix.shape[1]-1))
svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
latent_users = svd.fit_transform(interaction_matrix)
score_matrix = latent_users @ svd.components_
seen = interaction_matrix.to_numpy() > 0
print("Matrix shape:", interaction_matrix.shape, "| components:", n_components)

## ***8. Model Performance & Evaluation***

For every eligible user, the latest hotel is hidden. HitRate@5 is the fraction of
those hidden hotels that the model recovers among its five highest-ranked unseen
items. This evaluates ranking, not a numeric price prediction.

In [ ]:
user_ids = interaction_matrix.index.to_numpy()
hotel_keys = interaction_matrix.columns.to_numpy()
user_lookup = {int(user): pos for pos, user in enumerate(user_ids)}
item_lookup = {str(item): pos for pos, item in enumerate(hotel_keys)}
evaluable = test_events[test_events['userCode'].isin(user_lookup)]
hits = 0
for _, event in evaluable.iterrows():
    pos = user_lookup[int(event['userCode'])]
    scores = score_matrix[pos].copy()
    scores[seen[pos]] = -np.inf
    if item_lookup.get(str(event['hotel_key'])) in np.argsort(scores)[::-1][:5]:
        hits += 1
metrics = {
    'HitRate@5': hits / len(evaluable) if len(evaluable) else 0.0,
    'evaluable_users': len(evaluable),
    'users': interaction_matrix.shape[0],
    'hotel_items': interaction_matrix.shape[1],
    'components': n_components,
}
display(pd.Series(metrics).to_frame('value'))

### Evaluation interpretation

The hotel catalog is small in the supplied data, so the top-five score is an
educational baseline rather than a commercial guarantee. A production recommender
needs availability, dates, budget, explicit ratings, and click/conversion feedback
before it can rank real traveller choices reliably.

## ***9. MLOps & Deployment***

The project saves the SVD, score matrix, seen-item mask, hotel metadata, and
popularity fallback in one artifact. MLflow tracks the experiment, Airflow
re-trains it, Flask exposes a recommendation endpoint, and Docker/Kubernetes/Jenkins
provide a path to repeatable delivery.

# **Conclusion**

This notebook turns completed stays into a leakage-controlled hotel-ranking
baseline. Its most important evaluation choice is the last-interaction holdout,
which asks whether earlier history can recover the next hotel.

### ***Hurrah! You have successfully completed your Recommendation Capstone Project !!!***